# Automatisation des balises

Les documents XML que l'on passe par ce script doivent être formaté pour que chaque paragraphe soit sur une seule ligne. Ainsi les noms ne sont pas coupés par des retours à la ligne et des espaces qui pourraient empêcher la reconnaissance de certains noms lors de l'annotation automatique. 

## Chargement des bibliothèques

In [24]:
from bs4 import BeautifulSoup, NavigableString
import re
import csv
from joblib import Parallel, delayed

In [25]:
from os import mkdir
from pathlib import Path
DIR_V0 = Path('output/v0/')
OUTPUT_DIR = Path('output/Vbalise/')

OUTPUT_DIR.mkdir(parents=True, exist_ok=True) # Créer le dossier de sortie s'il n'existe pas déjà


## Création des fonctions

### Lire les fichiers csv et txt

Pour les 3 fonctions qui suivent on va lire les données que l'on veut annoter dans des documents csv et txt. 
Ils sont déjà existant dans le dossier `fichier_noms`. 

#### Fonction pour lire les noms d'auteurs à partir d'un fichier .csv

Le fichier `auteurs.csv` contient les noms de l'IndexPersonne que l'on peut annoter ou tenter d'annoter automatiquement.

Les colonnes sont nommées ainsi : `xml:id,Nom,NomITA,`


In [26]:
def lire_noms_indexPers(fichier_noms): #Définition d'une fonction pour lire les fichiers individuels avec les noms et les identifiants. En créer un dictionnaire avec une clé et une valeur. 
    noms_auteurs = [] #On créer une liste vide qui va contenir le dictionnaire.
    with open(fichier_noms, 'r', encoding='utf-8') as f: #ouvre le fichier qui sera renseigné en tant que fichier_noms
        reader = csv.DictReader(f)  # Utilise DictReader pour lire les noms de colonne
        for row in reader: #Pour chaque ligne dans les colonnes de reader qui est défini au dessus, on execute les manips qui suivent.
            nom = row['Noms'].strip()  # Récupérer la colonne 'Nom'
            id_noms = row['ID'].strip()  # Récupérer la colonne 'Id'
            if nom and id_noms: # Si on a bien un nom et un id on ajoute au dico (avec .append)
                noms = [n.strip() for n in nom.split(',') if n.strip()]
                noms_auteurs.append({'xml:id': id_noms, 'Nom': noms})  # Ajouter au dictionnaire avec 'id' et 'Nom'
    return noms_auteurs


#### Fonction pour lire les noms de lieu à partir d'un fichier .csv

Le fichier `lieux.csv` contient les lieux de l'IndexLieux que l'on peut annoter ou tenter d'annoter automatiquement.

Les colonnes sont nommées ainsi : `Continent,Pays,Nom,Id,NomIta,,`

Lorsqu'il y plusieurs possibilité pour un seul xml:id, ils sont tous dans la même "case" et seront prit en compte dans la liste comme : `{'id': 'Toulouse', 'Nom': ['Toulouse', 'Tolose']}`.

In [27]:
def lire_noms_indexLieux(fichier_lieux):
    noms_lieux = []
    with open(fichier_lieux, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f)  # Utilise DictReader pour lire les noms de colonne
        for row in reader:
            lieu = row['Noms'].strip()  # Récupérer la colonne 'Nom'
            id_lieu = row['ID'].strip()  # Récupérer la colonne 'Id'
            if lieu and id_lieu:
                lieux = [n.strip() for n in lieu.split(',') if n.strip()] # permet de récupérer tous les noms d'un lieu qui sont dans une seule case, séparés par une ","
                noms_lieux.append({'id': id_lieu, 'Nom': lieux})  # Ajouter au dictionnaire avec 'id' et 'Noms'
    
    return noms_lieux

Cette cellule ne sert qu'à vérifier la liste des lieux créée par la fonction lire_noms_lieux()

In [28]:
fichier_lieux = 'fichiers_noms/lieux.csv'
lire_noms_indexLieux(fichier_lieux)

[{'id': 'SaintRemyDeProvence', 'Nom': ['Saint Remi']},
 {'id': 'SaintEtienneVille', 'Nom': ['Saint Etienne']},
 {'id': 'SaintQuentinVille', 'Nom': ['Saint Quentin', 'San Quintino']},
 {'id': 'BassanoDelGrappa', 'Nom': ['Bassan']},
 {'id': 'SantAngeloInVado', 'Nom': ['Saint-Ange in Vado', 'Agnolo in Vado']},
 {'id': 'CittaDiCastello', 'Nom': ['Ville de Castello', 'Città di Castello']},
 {'id': 'Constantinople', 'Nom': ['Constantinople']},
 {'id': 'PordenoneVille', 'Nom': ['Pordenone', 'Pordenon']},
 {'id': 'ValleDiBlenio', 'Nom': ['Laval de Bregno']},
 {'id': 'Fontainebleau',
  'Nom': ['Fontaine-bleau', 'Fontainebleau', 'Fonteinebleau', 'Fontanableo']},
 {'id': 'Grottaferrata', 'Nom': ['Grotta Ferrata']},
 {'id': 'AixLaChapelle', 'Nom': ['Aix la Chapelle', 'Aix-la-Chapelle']},
 {'id': 'NeubourgDuche', 'Nom': ['Neubourg']},
 {'id': 'GambassiTerme', 'Nom': ['Cambassi']},
 {'id': 'MonteCavallo', 'Nom': ['Monte cavallo']},
 {'id': 'EmpireOrient', 'Nom': ['Empire d’Orient']},
 {'id': 'Thebes

### Fonction qui ajoute les balises `<persName>` et `<placeName>`

Cette fonction imite la fonctionnement de la fonction Rechercher/Remplacer en utilisant des expressions régulières.

/!\ Si le document contient déjà des balises, elles ne seront pas prises en compte. On peut alors se retrouver avec un document contentant des balises doublées. 
    De la même façon si on fait tourner un fichier qui a déjà été passé par ce notebook toutes les balises vont se dédoubler.

In [29]:
def ajouter_balise(texte, noms_lieux, noms_auteurs): #Définition de la fonction "rechercher" pour les mots qui sont dans les différents dictionnaires qui sont créer au dessus.
    
    # Remplacer chaque nom d'auteur par une balise <persName>
    for auteur in noms_auteurs: #Pour les auteurs (valeur du dictionnaire noms_auteurs)
        for pers in auteur['Nom'] :
            texte = re.sub(rf"(?<![#><\w]){pers}(?!<)\b", f'<persName ref="#{auteur["xml:id"]}">{pers}</persName>', texte)
        # re = utilisation des expressions régulières. 
        # .sub permet de remplacer avec les attribut sous cette forme là (valeur recherchée, remplacement, document cible)
        # (?<![#><\w]) Regarde la non présence des caractères entre crochet avant pers. (pour empêcher le balisage des noms dans les ref)
    
    # Remplacer chaque lieu par une balise <placeName>
    for lieu in noms_lieux:
        for nom in lieu['Nom']: # test pour toutes les possibilités de Nom dans la variable nom
            texte = re.sub(rf"(?<![#><\w]){nom}(?!<)\b", f'<placeName ref="#{lieu["id"]}">{nom}</placeName>', texte)

    
    return texte

#### Fonction pour ajouter les balises `<dates>`

In [30]:
def lire_date(texte, pattern):
    # Utiliser re.sub avec une fonction de remplacement
    def replacer(match):
        year = match.group()  # Extraire la correspondance
        return f'<date when="{year}">{year}</date>'
    
    # Appliquer re.sub pour remplacer toutes les correspondances
    texte = re.sub(pattern, replacer, texte)
    return texte

#### Manipuler du XML avec BeautifulSoup

Fonction qui utilise les autres fonctions créées ci-dessus, en lisant les fichiers d'entrés et en rajoutant les balises.  
La fonction remplace les caractères qui posent soucis (les chevrons).  
Enfin elle ouvre un fichier de sortie xml.  

In [31]:
def ajouter_balises_xml(fichier_xml, fichier_noms, fichier_lieux):
    noms_auteurs = lire_noms_indexPers(fichier_noms)
    noms_lieux = lire_noms_indexLieux(fichier_lieux)
    pattern = r"\b\d{4}\b"

    with open(fichier_xml, 'r', encoding='utf-8') as fichier:
        contenu_xml = fichier.read()
    print(f"Traitement de {fichier_xml.name}...")
    soup = BeautifulSoup(contenu_xml, 'xml')

    # Récupérer tous les nœuds texte non vides
    elements = [e for e in soup.find_all(string=True) if e.strip()]

    # Traiter chaque nœud texte en parallèle (regex uniquement)
    def traiter_element(element_str):
        nouveau_texte = ajouter_balise(element_str, noms_lieux, noms_auteurs)
        nouveau_texte = lire_date(nouveau_texte, pattern)
        return nouveau_texte

    resultats = Parallel(n_jobs=-1)(
        delayed(traiter_element)(str(element)) for element in elements
    )

    # Réinjecter séquentiellement (BeautifulSoup n'est pas thread-safe)
    for element, nouveau_texte in zip(elements, resultats):
        element.replace_with(NavigableString(nouveau_texte))

    fichier_modifie = soup.prettify()
    fichier_corrige = fichier_modifie.replace("&lt;", "<").replace("&gt;", ">")

    with open(OUTPUT_DIR / fichier_xml.name, 'w', encoding='utf-8') as fichier_out:
        fichier_out.write(fichier_corrige)

### Utilisation

#### Chargement dans des variables des fichiers csv. 

Les fichiers dont on a besoin pour l'annotation des noms et des lieux sont déjà indiqué dans la cellule suivante avec le bon chemin d'accès vers le dossier `fichiers_noms`. 

In [32]:
# On assigne les fichiers que l'on veut utiliser aux variables qui sont utilisées dans la fonction qui créé les dictionnaires. 
# On met alors le chemin de chacun des fichiers dans la bonne variable.
fichier_noms = 'fichiers_noms/auteurs.csv'
fichier_lieux = 'fichiers_noms/lieux.csv'

### Le fichier à annoter

On remplace le nom de fichier pour y accéder dans le dossier où il se trouve, tel que : `../Nom_du_fichier.xml`


In [33]:
#fichier_xml = 'exemple.xml'
#fichier_xml = '../script-preptxt/sortie.xml'
fichier_xml = '../Martin_DiscoursSongePoliphile.xml'

### Exécuter le programme

Cette section lance le balisage. La cellule suivante parcourt **séquentiellement tous les fichiers**
contenus dans `CORPUS_Index_Anais/Corpus_V0/` et invoque `ajouter_balises_xml()` pour chacun d'eux,
produisant des sorties dans le dossier `output/Vbalise`.

In [34]:
# Parcourt tous les fichiers XML de V0 et les balise en parallèle,
# en enregistrant le résultat dans le dossier Vbalise.
# n_jobs=-1 utilise tous les cœurs disponibles.
xml_files = sorted(DIR_V0.glob('*.xml'))

Parallel(n_jobs=-1)(
    delayed(ajouter_balises_xml)(xml_file, fichier_noms, fichier_lieux)
    for xml_file in xml_files
)

print(f"{len(xml_files)} fichier(s) traité(s) et sauvegardés dans '{OUTPUT_DIR}'.")

19 fichier(s) traité(s) et sauvegardés dans 'output\Vbalise'.
